# 深圳链家二手房数据采集与房价影响因素分析

**课程**：人工智能数据采集课程设计　|　**数据源**：[深圳链家二手房](https://sz.lianjia.com/ershoufang/)

## 项目目标
爬取深圳链家二手房列表页数据，清洗后从 **区域、面积、户型、朝向、装修、楼层、楼龄** 等维度分析房价影响因素，并实现 **关键字 / 商圈 / 标签 / 组合查询** 功能。

## 技术栈
| 环节 | 工具 |
|---|---|
| 发送请求 | requests（带 User-Agent + Cookie 反反爬） |
| 页面解析 | lxml + XPath |
| 数据存储 | pandas DataFrame → CSV |
| 清洗 | 正则表达式 re |
| 分析可视化 | matplotlib |
| 查询功能 | pandas 条件筛选（支持组合查询） |

## 整体流程
**请求网页 → XPath 解析 → 翻页循环爬取 → 存 CSV → 正则清洗 → 统计可视化 → 查询功能**

## 使用前必读（按顺序运行）
1. 先运行"环境准备"，再运行"配置与请求测试"，**必须先填入 Cookie**（获取方法见下一格说明）
2. 之后从上到下逐格运行；"交互式查询"一格会等待输入，属正常现象
3. ⚠️ 上传 GitHub / 交作业前，务必把 `COOKIE` 变量清空！

## 采集字段与清洗对照表

| 字段 | 页面原文示例 | 清洗后 |
|---|---|---|
| 标题 | 公园旁精装两房… | 原样保留 |
| 链接 | …/ershoufang/105122875.html | 原样保留（查重用） |
| 小区 / 商圈 | 旺业豪苑 / 东门 | 两列 |
| 房屋信息 | 1室2厅 \| 51.58平米 \| 西南 \| 精装 \| 低楼层(共33层) \| 2003年 \| 塔楼 | 正则拆成 **户型、面积、朝向、装修、楼层位置、总楼层、建成年份、楼型** 8 列 |
| 关注信息 | 2人关注 / 10个月以前发布 | **关注人数**、**发布时间** |
| 总价 | 190 | float（万元） |
| 单价 | 36,836元/平 | 去掉"元/平"和逗号 → int（元/㎡）★分析目标 |
| 标签 | 近地铁 \| VR房源 \| 随时看房 | 拼成一列，支持按标签查询 |

> 分析目标：**单价（元/㎡）**，其余字段均为候选影响因素。

## 第 0 步：环境准备

如果下一格报 `ModuleNotFoundError`，把 `!pip` 那行开头的 `#` 去掉运行一次即可。

In [3]:
# 如果缺库，去掉下一行开头的 # 运行一次
# !pip install requests lxml pandas matplotlib

import requests        # 发 HTTP 请求（相当于 Java 的 HttpClient）
import re              # 正则表达式，清洗字符串用
import time            # time.sleep 控制爬取速度
from lxml import etree # 把 HTML 解析成树，用 XPath 查找

import pandas as pd                 # 表格处理（内存里的 Excel）
import matplotlib.pyplot as plt     # 画图

# matplotlib 显示中文设置（Windows 自带微软雅黑）
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False   # 坐标轴负号正常显示

print('环境 OK')

环境 OK


## 第 1 步：配置 Cookie（反爬关键）

链家有"人机验证"反爬，纯程序请求会被拦截。解决：**把浏览器的 Cookie 借给程序**。

**获取 Cookie（3 步）**：
1. 用 Chrome/Edge 正常打开 https://sz.lianjia.com/ershoufang/ （能看到房源列表）
2. 按 `F12` → 顶部 **网络(Network)** → 按 `F5` 刷新 → 点第一条 `ershoufang` 请求
3. 右侧 **请求标头(Request Headers)** 里复制 `Cookie:` 后面的**整串**内容

> ⚠️ Cookie 等于你的上网身份：**交作业 / 上传 GitHub 前必须清空**；如果某天突然爬不动了，大概率是 Cookie 过期，重新复制一次即可。
> **实战经验（重要）**：① Cookie 必须用**刚从浏览器复制的新鲜值**——其中的 `hip` 令牌有时效，放久了必被拦；② 被拦截的表现是"网页长度 6000 左右 + 提示被拦截"，此时重新登录复制一次 Cookie 即可；③ 链家有限流：连续请求约 1-2 页后会拦一下，脚本已内置自动重试与冷却等待，属正常现象。

> **翻页被限流怎么办（实战总结）**：链家对连续请求限流严格（实测约每 2 分钟放行 1 页，被拦时返回约 6KB 的验证页）。解决办法：① 把翻页间隔 SLEEP 调大到 90~120 秒慢慢爬；② 用 Selenium 打开真实 Chrome 窗口，弹出人机验证时**人工拖滑块通过**，脚本检测到验证通过后自动继续下一页——本项目最终 220 条数据即用此法爬得（7 页 × 30 条 + 手动通过的 4 次验证）。


In [4]:
# ===== 配置与请求测试 =====

# ★★★ 已填入 Cookie；交作业 / 上传 GitHub 前务必清空这一行 ★★★
COOKIE = ''  # ← 使用前：从浏览器 F12 复制新鲜 Cookie 粘到引号内（严禁上传真实值）

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                  '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Cookie': COOKIE,
}

BASE_URL   = 'https://sz.lianjia.com/ershoufang/pg{}/'  # {} 处填页码，如 pg1、pg2
PAGE_START = 1     # 起始页
PAGE_END   = 5     # 爬到第几页（每页30套，5页=150套；正式爬可改成更大，最多100页）
SLEEP      = 8     # 每页之间停几秒：礼貌爬取，防止被封

assert COOKIE != '', '>>> 请先按上一格说明把 Cookie 填到 COOKIE 变量里！'

def fetch(url, retry=2):
    # 请求网页并返回 HTML；失败自动重试，被拦截返回 None
    for i in range(retry + 1):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=15)
            resp.encoding = 'utf-8'
            html = resp.text
            if 'sellListContent' in html:        # 真房源页才有这个列表容器
                return html
            print('  疑似被拦截(返回验证页)，状态码', resp.status_code, '，重试中...')
        except Exception as e:
            print('  请求异常:', e)
        time.sleep(12)                           # 失败后多等一会再试（链家限流冷却）
    return None

# 立即测试第 1 页
html1 = fetch(BASE_URL.format(1))
if html1:
    print('请求成功！网页长度', len(html1), '字符')
    with open('lianjia_page1.html', 'w', encoding='utf-8') as f:
        f.write(html1)
    print('已保存 lianjia_page1.html（调试用，可随时用 F12 对照）')
else:
    print('请求失败：Cookie 可能过期，重新复制一次再运行本格')

请求成功！网页长度 171960 字符
已保存 lianjia_page1.html（调试用，可随时用 F12 对照）


## 第 2 步：XPath 解析一个页面

原理：`etree.HTML()` 把 HTML 变成一棵树，XPath 就是"在树上按路径找节点"的查询语言。
核心两条规则：`//` 表示任意层级往下找，`text()` 取标签里的文字、`@href` 取属性、`[1]` 取第几个、`contains()` 模糊匹配。

> 下面的 XPath 已按真实页面结构写好并验证过；你想练习"Copy XPath → 裁剪"的话，随时可以换成自己复制的版本。

In [ ]:
# ===== 解析函数：一个页面 → 30 个字典（每套房源一个字典）=====

# 定位"每一套房源"。用 contains 匹配 clear：登录后页面会在 class 后面
# 追加 LOGVIEWDATA 等后缀（li class="clear LOGVIEWDATA..."），精确匹配会漏掉全部房源！
XPATH_ITEM = '//ul[contains(@class,"sellListContent")]/li[contains(@class,"clear")]'

# 每套房源内部的字段。开头的 . 表示"在这套房源内部找"
FIELDS = {
    '标题':     './/div[contains(@class,"title")]/a/text()',
    '链接':     './/div[contains(@class,"title")]/a/@href',
    '小区':     './/div[contains(@class,"positionInfo")]/a[1]/text()',
    '商圈':     './/div[contains(@class,"positionInfo")]/a[2]/text()',
    '房屋信息': './/div[contains(@class,"houseInfo")]/text()',
    '关注信息': './/div[contains(@class,"followInfo")]/text()',
    '总价(万)': './/div[contains(@class,"totalPrice")]//span/text()',
    '单价':     './/div[contains(@class,"unitPrice")]//span/text()',
    '标签':     './/div[contains(@class,"tag")]//span/text()',
}

def parse_page(html):
    # 解析一个列表页，返回字典列表
    tree  = etree.HTML(html)
    nodes = tree.xpath(XPATH_ITEM)
    result = []
    for n in nodes:                       # n = 一套房源的"小树"
        d = {}
        for name, xp in FIELDS.items():
            v = n.xpath(xp)               # xpath 返回值永远是列表
            v = [x.strip() for x in v if x and x.strip()]
            d[name] = ' | '.join(v)       # 多个值(如标签)用 | 拼成一格
        if d.get('总价(万)') and d.get('单价'):   # 再过滤一次缺价格的广告卡
            result.append(d)
    return result

# 用保存的第 1 页测试解析效果
df1 = pd.DataFrame(parse_page(html1))
print('第 1 页解析出', len(df1), '套（应为30）')
df1.head(3)

## 第 3 步：翻页循环爬取

把"请求 → 解析"包进 for 循环，URL 用 f-string 换页码：`pg1`、`pg2`…
每页之间 `time.sleep(SLEEP)` 控制速度——这是爬虫的"礼貌"，也是防封的关键。

In [ ]:
# ===== 翻页爬取主循环 =====

all_houses = []                       # 装所有页的结果
for pg in range(PAGE_START, PAGE_END + 1):
    print(f'正在爬第 {pg} 页 ...')
    html = fetch(BASE_URL.format(pg))
    if html is None:
        print(f'  第 {pg} 页失败，跳过')
        continue
    houses = parse_page(html)
    all_houses.extend(houses)         # 相当于 list.addAll
    print(f'  获得 {len(houses)} 套，累计 {len(all_houses)} 套')
    time.sleep(SLEEP)                 # 每页之间停 SLEEP 秒

df_raw = pd.DataFrame(all_houses)     # 列表[字典] 直接变表格，一个字典=一行
df_raw.to_csv('lianjia_raw.csv', index=False, encoding='utf-8-sig')  # utf-8-sig 保证 Excel 打开不乱码
print('原始数据已保存 → lianjia_raw.csv，共', len(df_raw), '条')
df_raw.head()

## 第 4 步：数据清洗（报告亮点之一）

爬下来的全是**字符串**，而且"房屋信息"一格里挤了 7 项信息。用**正则表达式**逐项提取，转成能计算的数字：

| 正则 | 从原文提取 | 变成 |
|---|---|---|
| `\d+室\d+厅` | 户型 | 文本 |
| `([\d.]+)平米` | 面积 | float |
| `精装\|简装\|毛坯` | 装修 | 文本 |
| `低楼层\|中楼层\|高楼层` | 楼层位置 | 文本 |
| `共(\d+)层` | 总楼层 | int |
| `(19\|20)\d{2}年` | 建成年份 | int → 再算**楼龄** |

In [ ]:
# ===== 数据清洗 =====

def clean(df):
    d = df.copy()
    h = d['房屋信息'].fillna('')

    # 从"房屋信息"里用正则逐项提取（不依赖固定顺序，缺项也不报错）
    d['户型']       = h.str.extract(r'(\d+室\d+厅)')[0]
    d['面积(㎡)']   = pd.to_numeric(h.str.extract(r'([\d.]+)平米')[0], errors='coerce')
    d['朝向']       = h.str.extract(r'(南北|东西|东南|西北|东北|西南|南|北|东|西)')[0]
    d['装修']       = h.str.extract(r'(精装|简装|毛坯|豪装|其他)')[0]
    d['楼层位置']   = h.str.extract(r'(地下室|低楼层|中楼层|高楼层)')[0]
    d['总楼层']     = pd.to_numeric(h.str.extract(r'共(\d+)层')[0], errors='coerce')
    d['建成年份']   = pd.to_numeric(h.str.extract(r'((?:19|20)\d{2})年')[0], errors='coerce')
    d['楼型']       = h.str.extract(r'(塔楼|板楼|板塔结合|平房)')[0]

    # 关注信息："2人关注 / 10个月以前发布"
    d['关注人数']   = pd.to_numeric(d['关注信息'].str.extract(r'(\d+)人关注')[0], errors='coerce')
    d['发布时间']   = d['关注信息'].str.split('/').str[1].str.strip()

    # 价格转数字
    d['总价(万)']   = pd.to_numeric(d['总价(万)'].str.replace(r'[^\d.]', '', regex=True), errors='coerce')
    d['单价(元/平)'] = pd.to_numeric(d['单价'].str.replace(r'\D', '', regex=True), errors='coerce')

    # 楼龄 = 当前年 - 建成年份
    this_year = pd.Timestamp.now().year
    d['楼龄'] = (this_year - d['建成年份']).clip(lower=0)

    keep = ['标题','链接','小区','商圈','户型','面积(㎡)','朝向','装修','楼层位置',
            '总楼层','建成年份','楼龄','楼型','关注人数','发布时间','总价(万)','单价(元/平)','标签']
    d = d[keep]
    d = d.dropna(subset=['单价(元/平)'])                 # 没单价的分析不了
    d = d.drop_duplicates(subset='链接')                 # 按链接去重
    return d.reset_index(drop=True)

df = clean(df_raw)
print('清洗后：', len(df), '条（清洗前', len(df_raw), '条）')
df.to_csv('lianjia_clean.csv', index=False, encoding='utf-8-sig')
print('已保存 → lianjia_clean.csv')
df.describe()   # 数值列统计总览

## 第 5 步：分析可视化——房价影响因素

围绕分析目标 **单价(元/平)**，逐一验证以下影响因素：
1. **地段**：哪个商圈贵？哪个商圈房源多？
2. **面积**：面积越大单价越低？（大面积通常单价更低）
3. **装修**：精装是否溢价？
4. **楼龄**：新房是否更贵？

In [ ]:
# ===== 图 A：地段（商圈）对房价的影响 =====
df_a = df[df['商圈'].fillna('') != '']                        # 去掉商圈为空的行
top_cnt = df_a['商圈'].value_counts().head(10)                # 房源数前10商圈
top_price = df_a.groupby('商圈')['单价(元/平)'].mean().sort_values(ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
top_cnt.sort_values().plot(kind='barh', ax=axes[0], color='#4C9BD6')
axes[0].set_title('房源数量 Top10 商圈')
axes[0].set_xlabel('在售套数')

top_price.sort_values().plot(kind='barh', ax=axes[1], color='#E8734A')
axes[1].set_title('平均单价 Top10 商圈')
axes[1].set_xlabel('平均单价（元/平）')
plt.tight_layout(); plt.show()

print('结论参考：房源最多的商圈是', top_cnt.index[0],
      '；最贵的商圈是', top_price.index[0],
      f'（均价 {top_price.iloc[0]:.0f} 元/平）')

In [ ]:
# ===== 图 B：面积对房价的影响 =====
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].scatter(df['面积(㎡)'], df['单价(元/平)'], alpha=0.35, s=18, color='#4C9BD6')
axes[0].set_xlabel('面积（㎡）'); axes[0].set_ylabel('单价（元/平）')
axes[0].set_title('面积 vs 单价 散点图')

bins   = [0, 50, 70, 90, 120, 150, 1000]
labels = ['0-50', '50-70', '70-90', '90-120', '120-150', '150+']
df['面积段'] = pd.cut(df['面积(㎡)'], bins=bins, labels=labels)
seg = df.groupby('面积段', observed=True)['单价(元/平)'].mean()
seg.plot(kind='bar', ax=axes[1], color='#5CB85C')
axes[1].set_title('各面积段平均单价')
axes[1].set_xlabel('面积段（㎡）'); axes[1].set_ylabel('平均单价（元/平）')
plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

print('各面积段均价：')
print(seg.round(0))

In [ ]:
# ===== 图 C：装修 / 楼型 / 楼龄对房价的影响 =====
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

df.groupby('装修')['单价(元/平)'].mean().sort_values().plot(kind='barh', ax=axes[0], color='#E8734A')
axes[0].set_title('不同装修的平均单价')

df.groupby('楼型')['单价(元/平)'].mean().sort_values().plot(kind='barh', ax=axes[1], color='#9B6BC9')
axes[1].set_title('不同楼型的平均单价')

age = df.dropna(subset=['楼龄']).copy()
age['楼龄段'] = pd.cut(age['楼龄'], [0, 10, 20, 30, 40, 100],
                      labels=['0-10年', '10-20年', '20-30年', '30-40年', '40年+'])
age.groupby('楼龄段')['单价(元/平)'].mean().plot(kind='bar', ax=axes[2], color='#4C9BD6')
axes[2].set_title('楼龄 vs 平均单价'); axes[2].set_xlabel('')
plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

print('装修均价：'); print(df.groupby('装修')['单价(元/平)'].mean().round(0))
print('楼型均价：'); print(df.groupby('楼型')['单价(元/平)'].mean().round(0))

## 第 6 步：查询功能（对应考核要求）

考核要求"按标题/标签/作者查询"，链家没有作者，对应调整为：**按关键字（标题/小区）、按商圈、按标签**查询，并额外支持**价格/面积组合查询**（评分点：操作便捷 + 结果美观）。

In [ ]:
# ===== 组合查询函数 + 美观输出 =====

def search(df, keyword=None, bizcircle=None, tag=None,
           min_total=None, max_total=None, min_area=None, max_area=None):
    # 组合查询：所有条件同时满足（条件之间是"且"），不填的条件自动忽略
    r = df
    if keyword:    r = r[r['标题'].str.contains(keyword, na=False) | r['小区'].str.contains(keyword, na=False)]
    if bizcircle:  r = r[r['商圈'] == bizcircle]
    if tag:        r = r[r['标签'].str.contains(tag, na=False)]
    if min_total is not None: r = r[r['总价(万)']   >= min_total]
    if max_total is not None: r = r[r['总价(万)']   <= max_total]
    if min_area  is not None: r = r[r['面积(㎡)']   >= min_area]
    if max_area  is not None: r = r[r['面积(㎡)']   <= max_area]
    return r.sort_values('单价(元/平)')

def show(r, n=8):
    # 美观输出：按单价渐变着色
    cols = [c for c in ['小区','商圈','户型','面积(㎡)','朝向','装修','楼层位置',
                        '总价(万)','单价(元/平)','标签'] if c in r.columns]
    r = r[cols].head(n)
    try:
        display(r.style.background_gradient(subset=['单价(元/平)'], cmap='OrRd'))
    except Exception:
        print(r.to_string(index=False))

# ---- 演示 1：关键字查询（标题/小区含"花园"）----
print('===== 演示1：关键字"花园" =====')
show(search(df, keyword='花园'))

# ---- 演示 2：标签查询（近地铁）----
print('===== 演示2：标签"近地铁" =====')
show(search(df, tag='近地铁'))

# ---- 演示 3：组合查询：东门商圈 + 100万以内 + 60平以上 ----
print('===== 演示3：组合查询（东门 + 总价<=100万 + 面积>=60㎡）=====')
show(search(df, bizcircle='东门', max_total=100, min_area=60))

### 交互式查询（可选运行）

下面是一个控制台式菜单，运行后**在输入框里输入数字**操作。输入 `0` 退出。
（注意：直接"运行全部"时会停在这里等输入，属正常现象）

In [ ]:
# ===== 交互式查询菜单 =====

while True:
    print()
    print('====== 链家二手房查询系统 ======')
    print('1. 按关键字查询（标题/小区）')
    print('2. 按商圈查询')
    print('3. 按标签查询（如：近地铁、VR房源、满五唯一）')
    print('4. 组合查询（关键字+商圈+价格区间）')
    print('0. 退出')
    choice = input('请输入编号: ').strip()

    if choice == '0':
        print('已退出， Bye~'); break
    elif choice == '1':
        show(search(df, keyword=input('输入关键字: ').strip()))
    elif choice == '2':
        print('可选商圈：', '、'.join(df['商圈'].dropna().unique()[:20]))
        show(search(df, bizcircle=input('输入商圈名: ').strip()))
    elif choice == '3':
        show(search(df, tag=input('输入标签: ').strip()))
    elif choice == '4':
        kw  = input('关键字(可回车跳过): ').strip() or None
        biz = input('商圈(可回车跳过): ').strip() or None
        lo  = input('最低总价/万(可回车跳过): ').strip()
        hi  = input('最高总价/万(可回车跳过): ').strip()
        show(search(df, keyword=kw, bizcircle=biz,
                    min_total=float(lo) if lo else None,
                    max_total=float(hi) if hi else None))
    else:
        print('无效编号，请重新输入')

## 附：直接使用已爬好的成品数据

第 3 步的翻页爬取受链家限流影响，现场演示可能只爬到少量页。项目附带已爬好的成品数据（`lianjia_clean.csv`，220 条，覆盖 69 个商圈），把下面单元格取消注释即可跳过爬取、直接进行第 4~6 步分析与查询。


In [ ]:
# 可选：读取随项目附带的成品数据（取消注释即可）
# df = pd.read_csv('lianjia_clean.csv')
# print('载入成品数据:', df.shape)
# df.head()

## 总结与报告要点

### 项目亮点（写报告 / 答辩直接用）
1. **反爬应对**：User-Agent 伪装 + 浏览器 Cookie 借用 + `sleep` 限速 + 失败自动重试 + 广告卡过滤
2. **正则清洗**：一条"房屋信息"字符串用正则拆出 8 个结构化字段，容错性强（缺项不报错）
3. **组合查询**：关键字/商圈/标签/价格/面积自由组合，结果按单价渐变着色，美观直观
4. **分析闭环**：从原始 HTML → CSV → 清洗 → 可视化结论，完整数据流水线

### 分析结论模板（跑完图表后自己填数字）
- 地段：最贵商圈是 ___（___ 元/平），约为最便宜商圈的 ___ 倍 → **地段是第一影响因素**
- 面积：___ ㎡ 段单价最高，大面积单价反而更低 → 单价与面积呈 ___ 相关
- 装修：精装比毛坯均价高 ___ 元/平
- 楼龄：楼龄越大单价整体越 ___

### 可选扩展（冲更高分）
- 爬详情页补充"挂牌次数、经纬度"等字段
- 用 sklearn 做线性回归，量化各因素对单价的影响系数

### ⚠️ 上传 GitHub / 交作业前必做
- 把配置格里 `COOKIE = ''` 清空（Cookie 等于个人身份，绝不能外传）
- 删除 `lianjia_page1.html` 等调试文件（可选）
- 建议在 README 放 2-3 张分析图 + 运行截图